In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, roc_auc_score


In [ ]:
df = pd.read_parquet("../data/processed/dataset_features.parquet")

df = df[df["condition"].isin(["EO", "EC"])]

df["y"] = df["condition"].map({"EO": 0, "EC": 1})


In [ ]:
NODE_FEATURES = [
    c for c in df.columns
    if c.startswith(("degree_", "clustering_", "betweenness_", "hub_"))
]


In [ ]:
META_FEATURES = ["age", "gender"]


In [ ]:
def evaluate_model(X, y, model):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "acc": "accuracy",
        "auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring
    )

    return {
        "accuracy_mean": scores["test_acc"].mean(),
        "auc_mean": scores["test_auc"].mean()
    }


In [ ]:
X = df[NODE_FEATURES].select_dtypes(include="number").dropna(axis=1)
y = df["y"]

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_lr)


In [ ]:
from sklearn.decomposition import PCA

pipe_pca_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_pca_lr)


In [ ]:
X_meta = pd.concat(
    [X, df[META_FEATURES]],
    axis=1
).dropna()

y_meta = df.loc[X_meta.index, "y"]

pipe_pca_meta = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X_meta, y_meta, pipe_pca_meta)
